<a href="https://colab.research.google.com/github/Calebchike/PVT-Extraction-Adjustment-Project/blob/Using-Dataframe/PVT_workbook_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import Modules**

In [1]:
import openpyxl
import pandas as pd
import numpy as np

# **Import Excel Data**

In [2]:
# Load the Excel file
file_path = "https://github.com/Calebchike/PVT-Extraction-Adjustment-Project/raw/Using-Dataframe/PVT_Workbook%20.xlsx"
all_sheets = pd.read_excel(file_path, sheet_name=None)

# See the names of all worksheets
print("Worksheet names:", all_sheets.keys())

# Convert all worksheets to DataFrame
summary_df = all_sheets['Summary']
cce_df = all_sheets['CCE']
dle_df = all_sheets['DLE']
sep_df = all_sheets['Separator Tests']


Worksheet names: dict_keys(['Summary', 'CCE', 'DLE', 'Separator Tests'])


# **Summary**

In [3]:
summary_df.head(5)

,Well information,Value
0,Initial Reservoir Pressure,4089.7
1,Reservoir Temperature,207.2
2,Reservoir Fluid mol%,52.9
3,Reservoir Fluid Mol Wt.,122.05
4,Bubble Point Pressure,988


# **Constant Composition Expansion (CCE)**

In [4]:
cce_df.head(5)

,Pressure,Relative Volume,Density,Y-Function,Compressibility
0,5500,0.9524,0.7663,NaN,0.000008
1,5000,0.9563,0.7632,NaN,0.000009
2,4500,0.9610,0.7595,NaN,0.000009
3,4090,0.9645,0.7567,NaN,0.000010
4,3500,0.9705,0.7520,NaN,0.000011


# **Differential Liberation Experiment (DLE)**

In [5]:
dle_df.head(5)

,Pressure,Rs,Bo,Oil Density,Bg,Gas Viscosity,Gas Density,Gas Z-Factor
0,5500,261.5,1.159,0.7663,NaN,NaN,NaN,NaN
1,5000,261.5,1.163,0.7632,NaN,NaN,NaN,NaN
2,4500,261.5,1.169,0.7595,NaN,NaN,NaN,NaN
3,4090,261.5,1.173,0.7567,NaN,NaN,NaN,NaN
4,3500,261.5,1.181,0.7520,NaN,NaN,NaN,NaN


# **DLE at Bubble Point Pressure**

In [6]:
bubble_point_pressure = summary_df.loc[summary_df['Well information'] == 'Bubble Point Pressure', 'Value'].iloc[0]
dle_Pb = dle_df[dle_df['Pressure'] == bubble_point_pressure]
dle_Pb

,Pressure,Rs,Bo,Oil Density,Bg,Gas Viscosity,Gas Density,Gas Z-Factor
9,988,261.5,1.217,0.7298,NaN,NaN,NaN,NaN


# **Optimum Sep & Adjusted PVT Parameters**

In [7]:
optimum_sep = sep_df.loc[sep_df['Bo'].idxmin()]
optimum_sep

,2
Test,3rd Sep Test
Pressure,102
Temp,90
Rs,233.24
API,42.27
Bo,1.1766


# **Correction Factors for Bo and Rs**

In [8]:
Bo_correction_factor = optimum_sep['Bo'] / dle_Pb['Bo'].item()
Rs_correction_factor = optimum_sep['Rs'] / dle_Pb['Rs'].item()

print(f"Bo correction factor: {Bo_correction_factor}")
print(f"Rs correction factor: {Rs_correction_factor}")

Bo correction factor: 0.9668036154478226
Rs correction factor: 0.8919311663479924


# **Adjusted PVT Table**

In [30]:
adjusted_pvt_df = pd.DataFrame({
    'Pressure': dle_df['Pressure'],
    'Bo_DLE': dle_df['Bo'],
    'Rs_DLE': dle_df['Rs'],
})

pb_value = dle_Pb['Pressure'].item()

# Create a Series of 'Relative Volume' from cce_df, indexed by 'Pressure'
# Then reindex it to match the 'Pressure' values in dle_df.
# Finally, reset the index to a default RangeIndex to align with adjusted_pvt_df.
cce_relative_volume_aligned = cce_df.set_index('Pressure')['Relative Volume'].reindex(dle_df['Pressure']).reset_index(drop=True)

# Define the condition for when to use CCE-based adjustment (Pressure > Bubble Point)
condition_cce_adjustment = adjusted_pvt_df['Pressure'] > pb_value

# Calculate Bo_adjusted using np.where for row-wise conditional logic
adjusted_pvt_df['Bo_adjusted'] = np.where(
    condition_cce_adjustment,
    adjusted_pvt_df['Bo_DLE'] * cce_relative_volume_aligned, # For P > Pb, use CCE Relative Volume
    adjusted_pvt_df['Bo_DLE'] * Bo_correction_factor         # For P <= Pb, use Bo_correction_factor
)

# Calculate Rs_adjusted using np.where for row-wise conditional logic
adjusted_pvt_df['Rs_adjusted'] = np.where(
    condition_cce_adjustment,
    adjusted_pvt_df['Rs_DLE'],                               # For P > Pb, Rs remains Rs_DLE
    adjusted_pvt_df['Rs_DLE'] * Rs_correction_factor         # For P <= Pb, use Rs_correction_factor
)

adjusted_pvt_df

,Pressure,Bo_DLE,Rs_DLE,Bo_adjusted,Rs_adjusted
0,5500,1.159,261.5,1.103832,261.500000
1,5000,1.163,261.5,1.112177,261.500000
2,4500,1.169,261.5,1.123409,261.500000
3,4090,1.173,261.5,1.131359,261.500000
4,3500,1.181,261.5,1.146161,261.500000
5,3000,1.187,261.5,1.158275,261.500000
6,2500,1.194,261.5,1.171911,261.500000
7,2000,1.201,261.5,1.185747,261.500000
8,1500,1.209,261.5,1.201142,261.500000
9,988,1.217,261.5,1.176600,233.240000
